# Roastmate — Training Model Roast (YOLOv8n-Cls)

Klasifikasi tingkat roasting biji kopi: **Green · Light · Medium · Dark**

- Dataset: DS003 (`gpiosenka/coffee-bean-dataset-resized-224-x-224`) [+ DS006 opsional]
- Arsitektur: YOLOv8n-cls (Ultralytics)
- Split: 70 / 15 / 15 (sebelum augmentasi)
- Output: `best.pt` + `best_int8.tflite`

Acuan: `docs/ARCHITECTURE_ML.md` §4–§5, §13 dan `ml/registries/experiment_registry.csv`.


## 1. Setup


In [ ]:
# 1. Setup — install dependency & cek GPU
!pip install -q ultralytics kagglehub

import os, sys, shutil, json, random
from pathlib import Path

import torch
import ultralytics

ultralytics.checks()
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Ambil dataset (DS003)

Di **Kaggle**: tambahkan dataset lewat panel **Add Input** (cari
`gpiosenka/coffee-bean-dataset-resized-224-x-224`), lalu sel di bawah akan
menemukannya otomatis di `/kaggle/input`.

Di luar Kaggle: otomatis fallback ke `kagglehub`.


In [ ]:
# 2. Ambil dataset DS003
import kagglehub

KAGGLE_INPUT = Path("/kaggle/input")
DS003_SLUG = "gpiosenka/coffee-bean-dataset-resized-224-x-224"

def find_dataset_dir():
    if KAGGLE_INPUT.exists():
        cands = [p for p in KAGGLE_INPUT.iterdir() if p.is_dir()]
        for p in cands:
            if "coffee" in p.name.lower():
                return p
        if cands:
            return cands[0]
    return Path(kagglehub.dataset_download(DS003_SLUG))

SRC = find_dataset_dir()
print("Sumber dataset:", SRC)
print("--- struktur (2 level teratas) ---")
for p in sorted(SRC.rglob("*")):
    depth = len(p.relative_to(SRC).parts)
    if depth <= 2:
        print("  " * depth + ("[D] " if p.is_dir() else "    ") + p.name)


## 3. Label mapping & bangun dataset terstruktur

Petakan nama folder asli ke kelas target. **Sesuaikan `LABEL_MAP` bila struktur
folder dataset berbeda** (lihat output sel sebelumnya).

> Catatan: DS006 tidak punya kelas `Green` (gap yang didokumentasikan di
> `ARCHITECTURE_ML.md` §2) — jadi training awal pakai DS003 saja.


In [ ]:
# 3. Bangun dataset terstruktur: roast_dataset/{train,val,test}/{kelas}
TARGET_CLASSES = ["Green", "Light", "Medium", "Dark"]

# Peta nama folder asli -> kelas target (lowercase, spasi jadi underscore)
LABEL_MAP = {
    "green": "Green", "light": "Light", "medium": "Medium", "dark": "Dark",
    "green_bean": "Green", "light_roast": "Light",
    "medium_roast": "Medium", "dark_roast": "Dark",
}

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
BUILD = WORK / "roast_dataset"

def detect_classes(root):
    found = {}
    for d in root.rglob("*"):
        if d.is_dir():
            key = d.name.lower().replace(" ", "_")
            if key in LABEL_MAP:
                found[key] = d
    return found

found = detect_classes(SRC)
print("Folder kelas terdeteksi:", {k: str(v) for k, v in found.items()})
assert found, "Tidak ada folder kelas yang cocok. Periksa struktur dataset & LABEL_MAP."

# Kumpulkan gambar per kelas target
per_class = {c: [] for c in TARGET_CLASSES}
for key, d in found.items():
    per_class[LABEL_MAP[key]].extend(
        [f for f in d.rglob("*") if f.suffix.lower() in IMG_EXT]
    )

# Split 70/15/15 SEBELUM augmentasi (aturan non-negotiable #3)
random.seed(42)
if BUILD.exists():
    shutil.rmtree(BUILD)

for c in TARGET_CLASSES:
    files = per_class[c]
    random.shuffle(files)
    n = len(files)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    parts = {
        "train": files[:n_train],
        "val": files[n_train:n_train + n_val],
        "test": files[n_train + n_val:],
    }
    for split, fl in parts.items():
        dst = BUILD / split / c
        dst.mkdir(parents=True, exist_ok=True)
        for f in fl:
            shutil.copy2(f, dst / f.name)
    print(f"{c:7s} total={n:5d}  train={len(parts['train']):5d}  "
          f"val={len(parts['val']):4d}  test={len(parts['test']):4d}")

print("\nDataset siap di:", BUILD)


## 4. Training

Augmentasi sengaja **ringan & tanpa geser hue**, karena warna = fitur utama
tingkat roasting (`ARCHITECTURE_ML.md` §4).


In [ ]:
# 4. Training YOLOv8n-cls
from ultralytics import YOLO

EPOCHS = 50
IMGSZ = 224
BATCH = 32
SEED = 42

model = YOLO("yolov8n-cls.pt")
results = model.train(
    data=str(BUILD),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    pretrained=True,
    project=str(WORK / "runs"),
    name="roast_yolov8n_cls",
    # Augmentasi ringan — jangan ubah warna ekstrem
    fliplr=0.5,
    flipud=0.0,
    degrees=10.0,
    translate=0.05,
    scale=0.1,
    hsv_h=0.0,   # tidak menggeser hue
    hsv_s=0.1,   # saturasi sangat ringan
    hsv_v=0.1,   # kecerahan sangat ringan
)
RUN_DIR = Path(results.save_dir)
print("Run dir:", RUN_DIR)


## 5. Evaluasi


In [ ]:
# 5. Evaluasi val & test
val_metrics = model.val(data=str(BUILD), split="val")
print("VAL  top1:", round(float(val_metrics.top1), 4), " top5:", round(float(val_metrics.top5), 4))

test_metrics = model.val(data=str(BUILD), split="test")
print("TEST top1:", round(float(test_metrics.top1), 4), " top5:", round(float(test_metrics.top5), 4))

print("Confusion matrix:", RUN_DIR / "confusion_matrix.png")
print("Fokus periksa: Light<->Medium dan Medium<->Dark (ARCHITECTURE_ML.md §13)")


## 6. Export ke TFLite INT8


In [ ]:
# 6. Export best.pt -> TFLite INT8 untuk mobile
best_pt = RUN_DIR / "weights" / "best.pt"
best_model = YOLO(str(best_pt))

tflite_path = best_model.export(format="tflite", int8=True, imgsz=IMGSZ, data=str(BUILD))
print("TFLite:", tflite_path)

# Salin artefak ke satu folder agar mudah diunduh
OUT = WORK / "roast_artifacts"
OUT.mkdir(exist_ok=True)
shutil.copy2(best_pt, OUT / "best.pt")
shutil.copy2(tflite_path, OUT / "best_int8.tflite")
print("Artefak:", [p.name for p in OUT.iterdir()])


## 7. Baris untuk `experiment_registry.csv`

Copy nilai di bawah ke `ml/registries/experiment_registry.csv` (satu baris).


In [ ]:
# 7. Cetak baris registry
import pandas as pd

row = {
    "experiment_id": "EXP-ROAST-001",
    "date": "ISI-TANGGAL",
    "model_task": "roast",
    "architecture": "yolov8n-cls",
    "datasets": "DS003",
    "classes": "Green,Light,Medium,Dark",
    "split": "70/15/15",
    "augmentation": "fliplr + degrees<=10 + hsv ringan",
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "acc_top1": round(float(val_metrics.top1), 4),
    "macro_f1": "",
    "test_acc_top1": round(float(test_metrics.top1), 4),
    "test_macro_f1": "",
    "confusion_notes": "fokus Light<->Medium, Medium<->Dark",
    "model_path": "ml/models/best_int8.tflite",
    "git_commit": "",
    "notes": "",
}
print(pd.Series(row).to_string())
